In [ ]:
!pip install pyrosettacolabsetup
import pyrosettacolabsetup; pyrosettacolabsetup.install_pyrosetta(cache_wheel_on_google_drive=False, serialization=False, mirror=0)
import pyrosetta; pyrosetta.init()
from pyrosetta import *
from pyrosetta.teaching import *


Note that USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE.
See https://github.com/RosettaCommons/rosetta/blob/main/LICENSE.md or email license@uw.edu for details.

Looking for compatible PyRosetta wheel file at google-drive/PyRosetta/colab.bin//wheels...

Resolving west.rosettacommons.org (west.rosettacommons.org)... 128.95.160.134, 128.95.160.135, 2607:4000:406::160:135, ...
Connecting to west.rosettacommons.org (west.rosettacommons.org)|128.95.160.134|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1665598342 (1.6G) [application/octet-stream]
--2026-09-11 12:23:35--  https://west.rosettacommons.org/pyrosetta/quarterly/release/pyrosetta-0-cp313-cp313-linux_x86_64.whl

HTTP request sent, awaiting response... 200 OK
Length: 1665598342 (1.6G) [application/octet-stream]
Saving to: ‘/PyRosetta/colab.bin/tmp41hcgajt/pyrosetta-0-cp313-cp313-linux_x86_64.whl’

pyrosetta-0-cp313-c 100%[===================>]   1.55G  41.9MB/s    in 47s     

202

In [ ]:
import os
import numpy as np
import pandas as pd

import pyrosetta
from pyrosetta import rosetta

import ipywidgets as widgets
from IPython.display import display, clear_output

from google.colab import files

In [ ]:
protocol = widgets.RadioButtons(
    options=[
        "Cartesian-ddG (protein stability)",
        "Flex ddG (protein-protein interface)"
    ],
    value="Cartesian-ddG (protein stability)",
    description="Protocol:",
    layout=widgets.Layout(width="500px")
)

display(protocol)

RadioButtons(description='Protocol:', layout=Layout(width='500px'), options=('Cartesian-ddG (protein stability…

In [ ]:
from google.colab import files
import os
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output

# These are created immediately, so they always exist
pdb_files = []
pdb_name_map = {}

upload_button = widgets.Button(
    description="Upload PDB files",
    button_style="info",
    icon="upload"
)

upload_output = widgets.Output()


def upload_pdbs(_):
    global pdb_files, pdb_name_map

    with upload_output:
        clear_output()

        uploaded = files.upload()

        # Reset for each new upload batch
        pdb_files.clear()
        pdb_name_map.clear()

        pdb_number = 1

        for original_name in uploaded.keys():

            if not original_name.lower().endswith(".pdb"):
                continue

            # Simple filename used internally by PyRosetta
            safe_name = f"input_{pdb_number:03d}.pdb"
            safe_path = os.path.join("/content", safe_name)

            shutil.copy2(original_name, safe_path)

            # Save path for PyRosetta
            pdb_files.append(safe_path)

            # Remember the real/original filename
            pdb_name_map[safe_path] = original_name

            pdb_number += 1

        print(f"\n{len(pdb_files)} PDB file(s) ready:\n")

        for path in pdb_files:
            print(
                f"{pdb_name_map[path]}  ->  "
                f"{os.path.basename(path)}"
            )


upload_button.on_click(upload_pdbs)

display(upload_button, upload_output)

Button(button_style='info', description='Upload PDB files', icon='upload', style=ButtonStyle())

Output()

In [ ]:
mutation_text = widgets.Textarea(
    value="A:2:A\nA:23:A\nA:34:A",
    placeholder=(
        "One mutation per line\n"
        "Example:\n"
        "A:23:A\n"
        "A:45:V"
    ),
    description="Mutations:",
    layout=widgets.Layout(
        width="600px",
        height="140px"
    )
)

display(mutation_text)

Textarea(value='A:2:A\nA:23:A\nA:34:A', description='Mutations:', layout=Layout(height='140px', width='600px')…

In [ ]:
def verify_inputs(_):

    with verify_output:
        clear_output()

        try:

            if not pdb_files:
                raise ValueError(
                    "Please upload at least one PDB file."
                )

            mutations = parse_mutations(
                mutation_text.value
            )

            print(
                f"Checking {len(pdb_files)} structures...\n"
            )

            for pdb_file in pdb_files:

                print("=" * 60)
                print(os.path.basename(pdb_file))
                print("=" * 60)

                try:

                    pose, checked = load_and_verify_pose(
                        pdb_file,
                        mutations
                    )

                    for m in checked:
                        print(
                            f"{m['chain']}:{m['pdb_resnum']} "
                            f"(Pose {m['pose_resnum']}): "
                            f"{m['wt']} -> {m['mutant']}"
                        )

                    print("OK\n")

                except Exception as e:

                    print("FAILED:")
                    print(e)
                    print()

        except Exception as e:

            print("ERROR:")
            print(e)

In [ ]:
valid_aas = set("ACDEFGHIKLMNPQRSTVWY")


def parse_mutations(text):

    mutations = []

    for line_number, line in enumerate(
        text.strip().splitlines(),
        start=1
    ):

        line = line.strip()

        # Ignore blank lines and comments
        if not line or line.startswith("#"):
            continue

        parts = line.split(":")

        if len(parts) != 3:
            raise ValueError(
                f"Line {line_number}: '{line}' is invalid. "
                "Expected Chain:Residue:MutantAA"
            )

        chain = parts[0].strip()
        resnum_text = parts[1].strip()
        mutant = parts[2].strip().upper()

        if len(chain) != 1:
            raise ValueError(
                f"Line {line_number}: "
                f"chain '{chain}' must be one character."
            )

        try:
            resnum = int(resnum_text)
        except ValueError:
            raise ValueError(
                f"Line {line_number}: "
                f"'{resnum_text}' is not a valid residue number."
            )

        if mutant not in valid_aas:
            raise ValueError(
                f"Line {line_number}: "
                f"'{mutant}' is not a valid amino acid."
            )

        mutations.append({
            "chain": chain,
            "pdb_resnum": resnum,
            "mutant": mutant
        })

    if not mutations:
        raise ValueError("No mutations were provided.")

    return mutations

In [ ]:
def load_and_verify_pose(pdb_file, mutations):

    pose = pyrosetta.pose_from_pdb(pdb_file)

    checked = []

    for mut in mutations:

        pose_num = pose.pdb_info().pdb2pose(
            mut["chain"],
            mut["pdb_resnum"]
        )

        if pose_num == 0:
            raise ValueError(
                f"Could not find "
                f"{mut['chain']}:{mut['pdb_resnum']} "
                f"in the structure."
            )

        wt = pose.residue(pose_num).name1()

        checked.append({
            **mut,
            "pose_resnum": pose_num,
            "wt": wt
        })

    return pose, checked

In [ ]:
print(pdb_files)
print(os.path.exists(pdb_files[0]))

['/content/1AB1.relaxed.best (4).pdb']
True


In [ ]:
verify_button = widgets.Button(
    description="Verify mutations",
    button_style="warning",
    icon="check"
)

verify_output = widgets.Output()

verify_button.on_click(verify_inputs)

display(verify_button)
display(verify_output)

Button(button_style='warning', description='Verify mutations', icon='check', style=ButtonStyle())

Output()

In [ ]:
from pyrosetta import rosetta
from pyrosetta.rosetta.protocols.ddg import CartesianddG

import numpy as np

In [ ]:
cart_scorefxn = pyrosetta.create_score_function("ref2015_cart")

core.scoring.etable: Starting energy table calculation
core.scoring.etable: smooth_etable: changing atr/rep split to bottom of energy well
core.scoring.etable: smooth_etable: spline smoothing lj etables (maxdis = 6)
core.scoring.etable: smooth_etable: spline smoothing solvation etables (max_dis = 6)
core.scoring.etable: Finished calculating energy tables.
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/HBPoly1D.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/HBFadeIntervals.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/HBEval.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/DonStrength.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/AccStrength.csv
basic.io.database: Database file opened: scoring/score_functions/rama/fd/all.ramaProb
basic.io.database: Database file opened: scor

In [ ]:
def get_neighbors(pose, center, radius=9.0):

    neighbors = rosetta.utility.vector1_unsigned_long()

    center_xyz = pose.residue(center).nbr_atom_xyz()

    for i in range(1, pose.total_residue() + 1):
        xyz = pose.residue(i).nbr_atom_xyz()

        if center_xyz.distance_squared(xyz) <= radius * radius:
            neighbors.append(i)

    return neighbors

In [ ]:
def make_cartesian_mutation(
    pose,
    pose_resnum,
    mutant_aa,
    iterations=5
):
    resnums = rosetta.utility.vector1_unsigned_long()
    resnums.append(pose_resnum)

    aa_vector = rosetta.utility.vector1_core_chemical_AA()

    # Convert one-letter code -> Rosetta AA enum
    aa_enum = rosetta.core.chemical.aa_from_oneletter_code(
        mutant_aa.upper()
    )

    aa_vector.append(aa_enum)

    mutation = CartesianddG.MutationSet(
        resnums,
        aa_vector,
        iterations
    )

    mutation.add_wildtypes(pose)

    return mutation

In [ ]:
def run_cartesian_ddg(
    pose,
    pose_resnum,
    mutant_aa,
    iterations=5,
    radius=9.0,
    bbnbrs=1
):

    mutation = make_cartesian_mutation(
        pose,
        pose_resnum,
        mutant_aa,
        iterations
    )

    neighbors = get_neighbors(
        pose,
        pose_resnum,
        radius
    )

    # Independent WT and mutant starting structures
    wt_pose = pose.clone()
    mut_pose = pose.clone()

    # Apply mutation
    CartesianddG.mutate_pose(
        mut_pose,
        mutation,
        cart_scorefxn
    )

    wt_scores = []
    mut_scores = []
    ddg_scores = []

    for iteration in range(iterations):

        # WT optimization
        CartesianddG.optimize_structure(
            mutation,
            cart_scorefxn,
            wt_pose,
            neighbors,
            False,      # flex_bb
            True,       # cartesian
            bbnbrs
        )

        # Mutant optimization
        CartesianddG.optimize_structure(
            mutation,
            cart_scorefxn,
            mut_pose,
            neighbors,
            False,      # flex_bb
            True,       # cartesian
            bbnbrs
        )

        wt_score = cart_scorefxn(wt_pose)
        mut_score = cart_scorefxn(mut_pose)

        ddg = mut_score - wt_score

        wt_scores.append(wt_score)
        mut_scores.append(mut_score)
        ddg_scores.append(ddg)

        print(
            f"    Iteration {iteration + 1}/{iterations}: "
            f"WT={wt_score:.3f}  "
            f"MUT={mut_score:.3f}  "
            f"ddG={ddg:.3f}"
        )

    return {
        "wt_scores": wt_scores,
        "mut_scores": mut_scores,
        "ddg_scores": ddg_scores,
        "mean_ddg": float(np.mean(ddg_scores)),
        "final_ddg": float(ddg_scores[-1]),
        "wt_pose": wt_pose,
        "mut_pose": mut_pose
    }

In [ ]:
cartesian_iterations = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description="Iterations:",
    continuous_update=False
)

cartesian_radius = widgets.FloatSlider(
    value=9.0,
    min=4.0,
    max=15.0,
    step=0.5,
    description="Radius (A):",
    continuous_update=False
)

cartesian_bbnbrs = widgets.IntSlider(
    value=1,
    min=0,
    max=3,
    step=1,
    description="bbnbrs:",
    continuous_update=False
)

display(
    cartesian_iterations,
    cartesian_radius,
    cartesian_bbnbrs
)

IntSlider(value=5, continuous_update=False, description='Iterations:', max=20, min=1)

FloatSlider(value=9.0, continuous_update=False, description='Radius (A):', max=15.0, min=4.0, step=0.5)

IntSlider(value=1, continuous_update=False, description='bbnbrs:', max=3)

In [ ]:
def run_ddg(_):

    global latest_results

    with run_output:
        clear_output()

        try:

            if not pdb_files:
                raise ValueError(
                    "Please upload at least one PDB file."
                )

            mutations = parse_mutations(
                mutation_text.value
            )

            results = []

            print(
                f"Processing {len(pdb_files)} PDB file(s)"
            )

            print(
                f"Protocol: {protocol.value}"
            )

            print()

            # ==========================================
            # LOOP OVER ALL PDB FILES
            # ==========================================

            for pdb_index, pdb_file in enumerate(
                pdb_files,
                start=1
            ):

                pdb_name = pdb_name_map.get(
                  pdb_file,
                  os.path.basename(pdb_file)
                )

                print()
                print("=" * 70)
                print(
                    f"PDB {pdb_index}/{len(pdb_files)}: "
                    f"{pdb_name}"
                )
                print("=" * 70)

                # --------------------------------------
                # Load structure and verify mutations
                # --------------------------------------

                try:

                    pose, checked = load_and_verify_pose(
                        pdb_file,
                        mutations
                    )

                except Exception as e:

                    print(f"Skipping {pdb_name}")
                    print(f"Reason: {e}")

                    continue

                # ======================================
                # CARTESIAN-ddG
                # ======================================

                if protocol.value.startswith(
                    "Cartesian"
                ):

                    # ----------------------------------
                    # LOOP OVER MUTATIONS
                    # ----------------------------------

                    for mutation_index, m in enumerate(
                        checked,
                        start=1
                    ):

                        mutation_name = (
                            f"{m['wt']}"
                            f"{m['pdb_resnum']}"
                            f"{m['mutant']}"
                        )

                        print()
                        print(
                            f"Mutation "
                            f"{mutation_index}/{len(checked)}: "
                            f"{m['chain']}:{mutation_name}"
                        )

                        try:

                            result = run_cartesian_ddg(
                                pose,
                                m["pose_resnum"],
                                m["mutant"],
                                iterations=(
                                    cartesian_iterations.value
                                ),
                                radius=(
                                    cartesian_radius.value
                                ),
                                bbnbrs=(
                                    cartesian_bbnbrs.value
                                )
                            )

                            results.append({

                                "PDB": pdb_name,

                                "Protocol":
                                    "Cartesian-ddG",

                                "Chain":
                                    m["chain"],

                                "PDB_residue":
                                    m["pdb_resnum"],

                                "Pose_residue":
                                    m["pose_resnum"],

                                "WT":
                                    m["wt"],

                                "Mutant":
                                    m["mutant"],

                                "Mutation":
                                    mutation_name,

                                "Mean_ddG_REU":
                                    result["mean_ddg"],

                                "Final_ddG_REU":
                                    result["final_ddg"],

                                "Status":
                                    "Success"
                            })

                            print(
                                f"  Mean ddG = "
                                f"{result['mean_ddg']:.3f} REU"
                            )

                        except Exception as e:

                            print(
                                f"  FAILED: {e}"
                            )

                            results.append({

                                "PDB": pdb_name,
                                "Protocol": "Cartesian-ddG",
                                "Chain": m["chain"],
                                "PDB_residue": m["pdb_resnum"],
                                "Pose_residue": m["pose_resnum"],
                                "WT": m["wt"],
                                "Mutant": m["mutant"],
                                "Mutation": mutation_name,
                                "Mean_ddG_REU": np.nan,
                                "Final_ddG_REU": np.nan,
                                "Status": f"Failed: {e}"
                            })


                # ======================================
                # FLEX-ddG
                # ======================================

                elif protocol.value.startswith(
                    "Flex"
                ):

                    print(
                        "Flex-ddG selected for "
                        f"{pdb_name}"
                    )

                    # Flex-ddG runner will go here
                    # once implemented.


            # ==========================================
            # RESULTS
            # ==========================================

            latest_results = pd.DataFrame(results)

            print()
            print("=" * 70)
            print("CALCULATIONS COMPLETE")
            print("=" * 70)

            if not latest_results.empty:

                display(latest_results)

                latest_results.to_csv(
                    "ddg_results.csv",
                    index=False
                )

                print(
                    "\nResults saved to ddg_results.csv"
                )

            else:

                print(
                    "No calculations were completed."
                )


        except Exception as e:

            print("ERROR:")
            print(type(e).__name__)
            print(e)

In [ ]:
run_button = widgets.Button(
    description="Run ΔΔG",
    button_style="success",
    icon="play"
)

run_output = widgets.Output()

run_button.on_click(run_ddg)

display(run_button)
display(run_output)

Button(button_style='success', description='Run ΔΔG', icon='play', style=ButtonStyle())

Output()